### 1. Tensors in Computer Memory
Computer memory is 1D (linear). Tensors are just a "view" on that linear block.
PyTorch uses **Row-major Order**: Consecutive elements of a row are adjacent in memory.

In [9]:
import torch 

tensor_2d = torch.tensor([[1, 2, 3, 4], [5, 6, 7 , 8], [9, 10 , 11, 12]])

print("Shape: ", tensor_2d.shape)

print("Stride:" , tensor_2d.stride())

print("Storage: ", tensor_2d.contiguous().view(-1))


Shape:  torch.Size([3, 4])
Stride: (4, 1)
Storage:  tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])


### 2. Sparse vs. Dense Tensors
* **Dense:** All elements are stored in contiguous memory (bad for high-entropy zeros).
* **Sparse (COO):** Stores only non-zero values and their coordinates (indices) to eliminate "silent computations" and reduce memory footprint.

In [14]:
rows, columns = 10000, 10000
dense_tensor = torch.zeros(rows, columns)

dense_tensor[0, 5] = 1.0

print("Dense tensor memory (MB): ", dense_tensor.element_size() * dense_tensor.numel() / 1e6)

indices = torch.tensor([[0], [5]])
values = torch.tensor([1.0])
sparse_tensor = torch.sparse_coo_tensor(indices, values, (rows, columns))

print(sparse_tensor)

Dense tensor memory (MB):  400.0
tensor(indices=tensor([[0],
                       [5]]),
       values=tensor([1.]),
       size=(10000, 10000), nnz=1, layout=torch.sparse_coo)


### 3. Tensor Broadcasting
Broadcasting automatically "stretches" the smaller tensor to match the larger one without physically duplicating data in memory.

In [15]:
A = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()

B = torch.tensor([10, 20, 30]).float()

C = A + B

print(C)

tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])


### 4. Data Normalization & Standardization
Unscaled data creates elliptical loss contours. Scaling transforms the loss surface into a spherical shape.

**Min-Max Normalization** (Maps to $[0, 1]$):
[cite_start]$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$ 

**Standardization / Z-Score** (Centers at $\mu=0$ with unit variance $\sigma=1$):
[cite_start]$$X_{scaled} = \frac{X - \mu}{\sigma}$$ 

In [23]:
data = torch.tensor([[25.0 , 50000.0], [30.0, 120000.0], [45.0, 80000.0]])

min_vals = data.min(dim=0).values
max_vals = data.max(dim=0).values

normalized_data = (data - min_vals) / (max_vals - min_vals)
print(normalized_data)

mean = data.mean(dim=0)
std = data.std(dim=0)

standardized_data = (data - mean) / std
print(standardized_data)

tensor([[0.0000, 0.0000],
        [0.2500, 1.0000],
        [1.0000, 0.4286]])
tensor([[-0.8006, -0.9492],
        [-0.3203,  1.0441],
        [ 1.1209, -0.0949]])


### 5. The Autograd Mechanism
PyTorch's automatic differentiation engine abstracts away manual calculus. It computes the Jacobian-Vector Product (JVP).

Let's find the minimum of the following non-linear function:
$$f(x) = x^2 - 10x + 25$$

*The analytical minimum is obviously at $x = 5$. Let's see if PyTorch can find it using only gradients.*

In [30]:
x = torch.tensor([0.0], requires_grad=True)

learning_rate = 0.1

for epoch in range(20):
    y = x**2 - 10*x + 25

    y.backward()

    with torch.no_grad():
        x -= learning_rate * x.grad
    x.grad.zero_()

print(f"\nFinal Result: {x.item():.4f} (Target: 5.0)")



Final Result: 4.9424 (Target: 5.0)
